In [6]:
#install textblob if not already installed using "pip install -U textblob"
from textblob import TextBlob
import nltk
# Download VADER, if not downloaded
#nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
# Sentiment analysis function for TextBlob tools
# create object for VADER sentiment function interaction
sia = SentimentIntensityAnalyzer()
def text_blob_sentiment(review, sub_entries_textblob):
    analysis = TextBlob(review)
    if analysis.sentiment.polarity >= 0.0001:
        if analysis.sentiment.polarity > 0:
            sub_entries_textblob['positive'] = sub_entries_textblob['positive'] + 1
            return 'Positive'

    elif analysis.sentiment.polarity <= -0.0001:
        if analysis.sentiment.polarity <= 0:
            sub_entries_textblob['negative'] = sub_entries_textblob['negative'] + 1
            return 'Negative'
    else:
        sub_entries_textblob['neutral'] = sub_entries_textblob['neutral'] + 1
        return 'Neutral'
    

# sentiment analysis function for VADER tool
def nltk_sentiment(review, sub_entries_nltk):
    vs = sia.polarity_scores(review)
    if not vs['neg'] > 0.05:
        if vs['pos'] - vs['neg'] > 0:
            sub_entries_nltk['positive'] = sub_entries_nltk['positive'] + 1
            return 'Positive'
        else:
            sub_entries_nltk['neutral'] = sub_entries_nltk['neutral'] + 1
            return 'Neutral'

    elif not vs['pos'] > 0.05:
        if vs['pos'] - vs['neg'] <= 0:
            sub_entries_nltk['negative'] = sub_entries_nltk['negative'] + 1
            return 'Negative'
        else:
            sub_entries_nltk['neutral'] = sub_entries_nltk['neutral'] + 1
            return 'Neutral'
    else:
        sub_entries_nltk['neutral'] = sub_entries_nltk['neutral'] + 1
        return 'Neutral'


In [7]:
# !pip install -q praw




In [14]:
from psaw import PushshiftAPI
import time
import datetime
import pandas as pd
import flair

api = PushshiftAPI()

def reddit_sentiment_2(search, start, end, max=10):
    start_date = time.mktime(datetime.datetime.strptime(start, "%d/%m/%Y").timetuple())
    end_date = time.mktime(datetime.datetime.strptime(end, "%d/%m/%Y").timetuple())

    posts = pd.DataFrame(columns=['title', 'flair', 'score', 'upvote_ratio', 'id',
                                'subreddit', 'url', 'num_comments', 'body', 'created'])  # Dataframe to store results

    while start_date < end_date:  # Continue loop until end date is reached
        S = api.search_submissions(subreddit=search,
                                    start=start_date, end=end_date, limit=max)  # Pull posts within date range
        for post in S:  # Looping through each post
            try: # Try/except to catch any erroneous post pulls

                if post.selftext != '[removed]' and post.selftext != '[deleted]': # Remove the deleted posts   

                    posts = posts.append(
                        {'title':post.title,
                          'flair':post.author_flair_css_class,
                          'score':post.score,
                          'upvote_ratio':post.upvote_ratio,
                          'id':post.id,
                          'subreddit':post.subreddit,
                          'url':post.url,
                          'num_comments':post.num_comments,
                          'body':post.selftext,
                          'created':datetime.datetime.fromtimestamp(post.created)}, ignore_index=True)  # Retrieve post data and append to dataframe
            except:
                next() # Continue loop if error is found

        if len(list(S)) < 100: # To identify when the last pull is reached
            break
        start_date = posts['created'].max()  # Select the next earliest date to pull posts from
        print(start_date)  # An indicator of progress

    ##run sentiment analysis
#     flair_sentiment = flair.models.TextClassifier.load('en-sentiment')  # Load model
#     posts['sentiment']= 0

#     sub_entries_textblob = {'negative': 0, 'positive' : 0, 'neutral' : 0}
#     sub_entries_nltk = {'negative': 0, 'positive' : 0, 'neutral' : 0}

#     for index, row in posts.iterrows():  # Iterate over the rows of the dataframe
#       s = flair.data.Sentence(row[0])  # Retrieve title of post
#       flair_sentiment.predict(s)  # Predict sentiment
#       posts['sentiment'][index] = s.labels[0] # Add sentiment to dataframe  

#       text_blob_sentiment(row[0], sub_entries_textblob)
#       nltk_sentiment(row[0], sub_entries_nltk)

#     return posts, sub_entries_textblob, sub_entries_nltk
    return posts


s = "20/09/2021"
e = "21/09/2021"
result = reddit_sentiment_2('TSLA', s, e, 20) #limit default 10
# result, textblob, nltk = reddit_sentiment_2('TSLA', s, e, 20) #limit default 10

# print('Overall Sentiment of topics by TextBlob :', textblob)
# print('Overall Sentiment of topics by VADER :',nltk)

# temp = result.sentiment.apply(lambda x: pd.Series(str(x).split(" ")))
# result['sentiment_result'] = temp[0]
# result['sentiment_score'] = temp[1]
# print('Overall flair result')
# result['sentiment_result'].value_counts()
print(done)

In [15]:

result.head()

,title,flair,score,upvote_ratio,id,subreddit,url,num_comments,body,created
0,Tesla short sellers are nowhere to be found af...,None,3,1.0,q1x0lb,TSLA,https://www.youtube.com/watch?v=E1WANjVTz6M,0,,2021-10-05 14:02:52
1,Serious question about Tesla’s full self driving,None,1,1.0,q18sn4,TSLA,https://www.reddit.com/r/TSLA/comments/q18sn4/...,1,What I’m trying to understand is if $TSLA has ...,2021-10-04 16:20:48
2,YIKES --- AKL motorways are stressful at the m...,None,1,1.0,q0waam,TSLA,https://youtu.be/Xd_pNVy_Mss,0,,2021-10-04 02:19:48
3,Top 10 Trending Stocks on Reddit - TSLA at #6,None,1,1.0,q0jrmj,TSLA,https://www.reddit.com/r/TSLA/comments/q0jrmj/...,1,"Hey Everyone,\n\nWanted to provide an update f...",2021-10-03 15:17:09
4,Carpool inequities,None,1,1.0,pzy11i,TSLA,https://www.reddit.com/r/TSLA/comments/pzy11i/...,0,So I drive in a carpool to get my kids to scho...,2021-10-02 16:07:46


In [138]:
result.to_csv("TSLA_reddit_post.csv",index=False)

In [139]:
from psaw import PushshiftAPI
import time
import datetime

import flair

api = PushshiftAPI()

def reddit_sentiment_2(search, start, end, max=10):
    start_date = time.mktime(datetime.datetime.strptime(start, "%d/%m/%Y").timetuple())
    end_date = time.mktime(datetime.datetime.strptime(end, "%d/%m/%Y").timetuple())

    comments = pd.DataFrame(columns=['parent_id', 'flair', 'score', 'id',
                                'subreddit', 'url', 'body', 'created'])  # Dataframe to store results

    while start_date < end_date:  # Continue loop until end date is reached
        S = api.search_comments(subreddit=search,
                                    start=start_date, end=end_date, limit=max)  # Pull posts within date range
        for comment in S:  # Looping through each post
            try: # Try/except to catch any erroneous post pulls

                if comment.body != '[removed]' and comment.body != '[deleted]': # Remove the deleted posts             
                    comments = comments.append(
                        {'body':comment.body,
                        'flair':comment.author_flair_css_class,
                        'score':comment.score,
                        'id':comment.id,
                        'subreddit':comment.subreddit,
                        'parent_id':comment.parent_id,
                        'created':datetime.datetime.fromtimestamp(comment.created)}, ignore_index=True)  # Retrieve post data and append to dataframe
            except:
                next() # Continue loop if error is found

        if len(list(S)) < 100: # To identify when the last pull is reached
            break
        start_date = comments['created'].max()  # Select the next earliest date to pull posts from
        print(start_date)  # An indicator of progress

    ##run sentiment analysis
    flair_sentiment = flair.models.TextClassifier.load('en-sentiment')  # Load model
    comments['sentiment']= 0

    sub_entries_textblob = {'negative': 0, 'positive' : 0, 'neutral' : 0}
    sub_entries_nltk = {'negative': 0, 'positive' : 0, 'neutral' : 0}

    for index, row in comments.iterrows():  # Iterate over the rows of the dataframe
        s = flair.data.Sentence(row[0])  # Retrieve title of post
        flair_sentiment.predict(s)  # Predict sentiment
        comments['sentiment'][index] = s.labels[0] # Add sentiment to dataframe  

        text_blob_sentiment(row[0], sub_entries_textblob)
        nltk_sentiment(row[0], sub_entries_nltk)

    return comments, sub_entries_textblob, sub_entries_nltk


s = "20/09/2021"
e = "21/09/2021"
result, textblob, nltk = reddit_sentiment_2('TSLA', s, e, 100000) #limit default 10

print('Overall Sentiment of topics by TextBlob :', textblob)
print('Overall Sentiment of topics by VADER :',nltk)

temp = result.sentiment.apply(lambda x: pd.Series(str(x).split(" ")))
result['sentiment_result'] = temp[0]
result['sentiment_score'] = temp[1]
print('Overall flair result')
result['sentiment_result'].value_counts()

C:\Users\Shawn\anaconda3\lib\site-packages\psaw\PushshiftAPI.py:252: UserWarning: Not all PushShift shards are active. Query results may be incomplete
  warnings.warn(shards_down_message)


2021-09-28 20:20:13,400 loading file C:\Users\Shawn\.flair\models\sentiment-en-mix-distillbert_4.pt


<ipython-input-139-2e0ad84d9432>:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comments['sentiment'][index] = s.labels[0] # Add sentiment to dataframe


Overall Sentiment of topics by TextBlob : {'negative': 0, 'positive': 0, 'neutral': 5386}
Overall Sentiment of topics by VADER : {'negative': 0, 'positive': 0, 'neutral': 5386}
Overall flair result


NEGATIVE    4893
POSITIVE     493
Name: sentiment_result, dtype: int64

In [140]:
result.to_csv("TSLA_reddit_comment.csv",index=False)

In [17]:
import praw

reddit = praw.Reddit(client_id='QnYrWKGOVJf2yohYMZgy2A',
                     client_secret='VskYCGxQCoVYytrFCWVciAl6Q9OzlA',
                     user_agent='g3t5')


# replication of comment section of reddit post
def replies_of(top_level_comment, count_comment):
    if len(top_level_comment.replies) == 0:
        count_comment = 0
        return
    else:
        for num, comment in enumerate(top_level_comment.replies):
            try:
                count_comment += 1
                print('-' * count_comment, comment.body)
            except:
                continue
            replies_of(comment, count_comment)
            
def reddit_sentiment(search):
  # get 10 hot posts
    top_posts = reddit.subreddit(search).top('week', limit=5)
    for submission in top_posts:
        try :
            print(submission.subreddit)
            replies_of(top_level_comment,count_comm)
        except:
            continue


reddit_sentiment("AAPL")

AAPL
AAPL
AAPL
